In [ ]:
# If neccesary install once:
#!pip install ipyleaflet ipywidgets
from ipyleaflet import Map, TileLayer, DrawControl
from ipywidgets import HTML, VBox
from energymap4py import data_access as em_data

# EnergyMap data from an OSM map
## This example shows how multiple buildings can be selected interactively on an OSM map using a polyline or polygon tool, for which information can then be retrieved from the EnergyMap database.

In [2]:
# Helping functions for catching coordinates of selected polylines or polygons from OSM 
def _store_line_from_geojson(geo_json, source="callback"):
    global poly_coords_ll, poly_geojson, polygon_coords_ll, polygon_geojson, info
    if not geo_json:
        return
    geom = geo_json.get("geometry", {})
    if geom.get("type") == "LineString":
        poly_geojson = geo_json
        poly_coords_ll = geom["coordinates"]  # GeoJSON-order: [lon, lat]
        info.value = f"<b>Polyline defined:</b> ({len(poly_coords_ll)} Points, {source})"
    if geom.get("type") == "Polygon":
        # GeoJSON-Polygone: List of rings -> [exterior, hole1, hole2, ...]
        rings = geom.get("coordinates", [])
        if rings:
            polygon_coords_ll = rings[0] # outer ring (lon, lat)
            polygon_geojson = geo_json
            info.value = f"<b>Polygon defined:</b> ({len(polygon_coords_ll)} Points, {source})."
        else:
            info.value = "Got empty polygon."
    else:
        info.value = HTML("Please draw a <b>Polyline</b> or a <b>Polygon</b>. Click, for defining coordinates – <b>double click</b> finishing polyline or close polygon.")

# 1) Compatible on_draw: catches (action, geo_json) AND (target, action, geo_json)
def on_draw(*args, **kwargs):
    if len(args) == 2:
        action, geo_json = args
    elif len(args) == 3:
        _, action, geo_json = args
    else:
        action = kwargs.get("action")
        geo_json = kwargs.get("geo_json")
    _store_line_from_geojson(geo_json, source="on_draw")

# 2) Fallback/Sync: observing feature-collection-state
def _sync_from_data(change):
    fc = change["new"]
    if not fc or not fc.get("features"):
        return
    last = fc["features"][-1]
    _store_line_from_geojson(last, source="observe")


## OSM Map Setup

In [39]:
poly_coords_ll = []   # coordinate list of the polyline: [[lon, lat], ...]
poly_geojson = None   # GeoJSON feature (LineString)

polygon_coords_ll = []  # coordinate list of the polygon: [[lon, lat], ...]
polygon_geojson = None  # GeoJSON feature (Polygon)

m = Map(center=(52.532461318193775,13.422214655200957), zoom=17, scroll_wheel_zoom=True)
m.add_layer(TileLayer(url="https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png", name="OSM"))
draw = DrawControl(polyline={"shapeOptions": {"weight": 4}}, polygon={"shapeOptions": {"weight": 4}})
# deactivate tools by empty dicts !
#draw.polygon = {}
#draw.polyline = {}
draw.circle = {}
draw.rectangle = {}
draw.marker = {}
draw.circlemarker = {}

draw.on_draw(on_draw)
draw.observe(_sync_from_data, names="data")
m.add_control(draw)

info = HTML("Please draw a <b>Polyline</b> or a <b>Polygon</b>. Click, for defining coordinates – <b>double click</b> finishing polyline or close polygon.")
VBox([m, info])


## Information about all buildings from the EnergyMap database 20 m left and right of the polyline

In [ ]:
coords = []
res1 = []
if poly_coords_ll != []:
    for i in range(len(poly_coords_ll)):
        coords.append((poly_coords_ll[i][0],poly_coords_ll[i][1]))    
    # catch building data from the server 
    res1 = em_data.by_line(line_points=coords, dist=20)
    for r in res1:
        print('uuid:',r['uuid'],'street:',r['street_name'],'age:',r['year_of_construction'],'heat present:',r['cons_c1r1'],'[kWh/a]','spec. heat present:',r['spec_cons_c1r1'],'[kWh/m2a]')
else:
    print("Draw a polyline on the map to select buildings of your interest !")


## Information about all buildings from the EnergyMap database which are completely located within the polygon

In [ ]:
coords = []
res2 = []
if polygon_coords_ll != []:
    for i in range(len(polygon_coords_ll)):
        coords.append((polygon_coords_ll[i][0],polygon_coords_ll[i][1]))
    # catch building data from the server 
    res2 = em_data.by_polygon(polygon_points=coords)
    for r in res2:
        print('uuid:',r['uuid'],'street:',r['street_name'],'age:',r['year_of_construction'],'heat present:',r['cons_c1r1'],'[kWh/a]','spec. heat present:',r['spec_cons_c1r1'],'[kWh/m2a]')
else:
    print("Draw a polygon on the map to select buildings of your interest !")


## Data postprocessing and evaluation

In [38]:
if res1 != [] and res2 != []:
    totalHeat = [float(r['cons_c1r1']) for r in res1] + [float(r['cons_c1r1']) for r in res2]
    heatedArea = [float(r['nrf_area']) for r in res1] + [float(r['nrf_area']) for r in res2]
    buildingAges = [float(r['year_of_construction']) for r in res1] + [float(r['year_of_construction']) for r in res2]
    print("The present heating demand of the", len(totalHeat), "selected buildings is", int(sum(totalHeat)/1000), "MWh/a.")
    print("Together they have a total heated area of", int(sum(heatedArea)), "m2.")
    print("The oldest building is from", int(min(buildingAges)), "and the newest from", int(max(buildingAges)),".")
else:
    print("Draw a polyline and a polygon on the map to select buildings of your interest !")


The present heating demand of the 28 selected buildings is 4904 MWh/a.
Together they have a total heated area of 36875 m2.
The oldest building is from 1879 and the newest from 1976 .
